## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **27da30f0-699d-4321-92f4-9025d9b2eff2**


## Dataset

# **Analyse climatique 2020 - Vitesse et direction du vent nocturne, résolution 200m, 2045-2074**


## Description

Les données montrent le champ de vent dans le canton de Genève qui a été modélisé avec le modèle climatique urbain FITNAH-3D. Les résultats sont basés sur une résolution spatiale horizontale de 200 m et sur une situation météorologique estivale autochtone &#40;condition météorologique de rayonnement à vent faible&#41;. Ils sont valables pour la zone de résidence humaine à 2 m du sol et représentent la situation de nuit à 4:00 heures &#40;refroidissement maximal&#41;. Le paramètre climatique est considéré à la fois dans sa caractéristique actuelle et future &#40;y compris le changement climatique et le développement futur&#41;. Les augmentations de température supposées pour les périodes futures se réfèrent aux résultats des scénarios climatiques pour la Suisse &#40;CH2018&#41;. Les hausses de température correspondent au signal de changement estival moyen du scénario RCP 8.5 pour l&#x27;Ouest de la Suisse. Attention: la direction correspond à la direction géographique, pas météorologique&#33; &#40;par exemple dans ce cas 0° correspond à un vent du sud vers le nord&#41; Période climatique: P2: période future 2045-2074 &#40;&quot;2060&quot;&#41; -&gt; + 2,6 °C par rapport à P0 Scénario de développement: P1, P2 et P3: occupation des sols avec les projets qui devraient être réalisés d&#x27;ici 2035 Pour les zones hors canton, les données de base relatives à la couverture du sol ont une précision inférieure à Genève. Les arbres isolés et les surfaces d&#x27;imperméabilisation autour des bâtiments, deux facteurs qui influencent les paramètres climatiques, n&#x27;ont pas pu être intégrés au modèle d&#x27;analyse. Il s&#x27;agit dès lors de considérer les données hors canton avec prudence. Rappel : conformément aux CGU, les informations relatives à l&#x27;analyse climatique sont sans valeur juridique.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/analyse-climatique-2020-vitesse-et-direction-du-vent-nocturne-resolution-200m-2045-20741)


## Metadata

- **identifier:** CLIMAT&#95;VENT&#95;200M&#95;4H00&#95;P2@canton-geneve
- **publisher.name:** Direction de la durabilité et du climat
- **organization.display&#95;name:** Kanton Genf
- **organization.url:** 
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** 
- **issued:** 2025-06-12 00:00:00
- **metadata&#95;created:** 2026-01-12T16:00:27.730713
- **metadata&#95;modified:** 2026-10-31T03:41:29.789390
- **Resource id:** 5ba0dad3-f4c7-467e-b960-e26549b391a7
- **Resource name:** CSV
- **Resource package&#95;id:** 27da30f0-699d-4321-92f4-9025d9b2eff2
- **Resource description:** Service téléchargement extracteur
- **Resource issued:** 2025-06-12 00:00:00
- **Resource modified:** 2026-10-29 00:00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;open
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;open


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **CSV** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://extract.sitg.ge.ch/files/CLIMAT_VENT_200M_4H00_P2-csv.zip")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** remy.zinder@etat.ge.ch &#124; Direction de la durabilité et du climat
